# Varying Slopes and Correlated Group Effects

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain when and why **varying slopes** are needed in addition to varying intercepts.
2. Write down the **multivariate normal** prior for correlated group-level intercepts and slopes.
3. Decompose the covariance matrix $\boldsymbol{\Sigma}$ into scales ($\boldsymbol{\tau}$) and correlations ($\boldsymbol{\Omega}$), and explain *why* this decomposition is used.
4. Define the **LKJ prior** on correlation matrices and explain what its concentration parameter $\eta$ controls.
5. Derive the **Cholesky decomposition** and explain why it makes MCMC more efficient.
6. Fit a varying-intercept-varying-slope model in **PyMC** and interpret the learned correlations.

### Prerequisites

- Notebook 02 (varying intercepts, centered/non-centered, funnel geometry).
- Linear algebra: matrix multiplication, positive-definite matrices, Cholesky factorisation.

In [ ]:
import sys, os, shutil
from pathlib import Path

os.environ["PYTENSOR_FLAGS"] = "device=cpu,floatX=float64,cxx="

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        "sudo apt-get update -qq && sudo apt-get install -y -qq "
        "libcairo2-dev libpango1.0-dev && pip install -q manim pymc arviz ipython==8.21.0"
    )

_miktex_bin = Path.home() / "AppData/Local/Programs/MiKTeX/miktex/bin/x64"
if _miktex_bin.exists() and str(_miktex_bin) not in os.environ.get("PATH", ""):
    os.environ["PATH"] += os.pathsep + str(_miktex_bin)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, os.path.abspath("../../src"))
from amstats.plotting import apply_style

apply_style()

HAS_PYMC = False
try:
    import pymc as pm
    import arviz as az
    HAS_PYMC = True
    print(f"PyMC {pm.__version__}, ArviZ {az.__version__}")
except Exception as e:
    print(f"PyMC not available: {type(e).__name__}: {e}")


class Cfg:
    root = Path("../../").resolve()
    gif_dir = root / "media" / "gifs"
    has_latex: bool = (
        shutil.which("latex") is not None or shutil.which("pdflatex") is not None
    )

    def __init__(self):
        self.gif_dir.mkdir(parents=True, exist_ok=True)

    def apply_manim_config(self):
        from manim import config as mcfg
        mcfg.format = "gif"

    def math_text(self, expr, **kwargs):
        from manim import MathTex, Text
        if self.has_latex:
            return MathTex(expr, **kwargs)
        return Text(expr, **kwargs)

    def save_gifs(self, clean=True):
        local_media = Path("media")
        found = list(local_media.rglob("*.gif")) if local_media.exists() else []
        if not found:
            print("  No new GIFs to save.")
            return
        for gif in found:
            dest = self.gif_dir / gif.name
            shutil.copy2(gif, dest)
            print(f"  \u2713 media/gifs/{gif.name}")
        if clean:
            for sub in ("videos", "images", "Tex"):
                d = local_media / sub
                if d.exists():
                    shutil.rmtree(d, ignore_errors=True)
            print("  Cleaned up local temp render files (kept media/jupyter/).")


cfg = Cfg()
rng = np.random.default_rng(42)

---

## 1. When Intercepts Alone Are Not Enough

In the varying-intercept model from the previous notebook, groups differ in their **baseline** but respond to predictors identically:

$$y_i = \alpha_{j[i]} + \beta \, x_i + \varepsilon_i$$

This assumes parallel regression lines — same slope, different heights.  But in many real settings, the *effect* of a predictor varies across groups:

- The effect of **class size** on test scores may differ between schools with different resources.
- The effect of **treatment dose** on recovery may differ between hospitals with different patient populations.
- The effect of **urbanisation** on contraception use may differ between districts with different cultural norms.

When this happens, we need **varying slopes**.

---

## 2. The Varying-Intercept, Varying-Slope Model

### The Model

We now allow both the intercept and slope to vary by group:

**Level 1 (likelihood):**

$$y_i \mid \alpha_{j[i]}, \beta_{j[i]}, \sigma \;\sim\; \mathcal{N}\!\left(\alpha_{j[i]} + \beta_{j[i]} \, x_i, \;\; \sigma^2\right)$$

**Level 2 (group-level):**

$$\begin{pmatrix} \alpha_j \\ \beta_j \end{pmatrix} \;\sim\; \text{MVNormal}\!\left(\begin{pmatrix} \bar{\alpha} \\ \bar{\beta} \end{pmatrix}, \;\; \boldsymbol{\Sigma}\right)$$

The critical new element is the $2 \times 2$ **covariance matrix** $\boldsymbol{\Sigma}$.  It captures:
- How much intercepts vary ($\Sigma_{11} = \tau_\alpha^2$).
- How much slopes vary ($\Sigma_{22} = \tau_\beta^2$).
- **Whether intercepts and slopes are correlated** ($\Sigma_{12} = \rho \, \tau_\alpha \, \tau_\beta$).

### Why Correlation Matters

Consider schools where the intercept $\alpha_j$ represents the baseline test score and $\beta_j$ represents the effect of class size.  If schools with high baseline scores (well-funded) also have a *weaker* class-size effect (because they have other resources), then $\rho < 0$.  Ignoring this correlation means losing information that helps estimate both $\alpha_j$ and $\beta_j$.

A model with **independent** varying intercepts and slopes is a special case where $\rho = 0$.  The correlated model is strictly more general, and the data will inform whether $\rho$ is needed.

---

## 3. Decomposing the Covariance Matrix

We do **not** put a prior directly on $\boldsymbol{\Sigma}$.  Instead, we decompose it into interpretable components:

$$\boldsymbol{\Sigma} = \text{diag}(\boldsymbol{\tau}) \;\boldsymbol{\Omega}\; \text{diag}(\boldsymbol{\tau})$$

where:
- $\boldsymbol{\tau} = (\tau_\alpha, \tau_\beta)^\top$ are the **scale** (standard deviation) parameters.
- $\boldsymbol{\Omega}$ is the $2 \times 2$ **correlation matrix**:

$$\boldsymbol{\Omega} = \begin{pmatrix} 1 & \rho \\ \rho & 1 \end{pmatrix}$$

Written out:

$$\boldsymbol{\Sigma} = \begin{pmatrix} \tau_\alpha & 0 \\ 0 & \tau_\beta \end{pmatrix} \begin{pmatrix} 1 & \rho \\ \rho & 1 \end{pmatrix} \begin{pmatrix} \tau_\alpha & 0 \\ 0 & \tau_\beta \end{pmatrix} = \begin{pmatrix} \tau_\alpha^2 & \rho \tau_\alpha \tau_\beta \\ \rho \tau_\alpha \tau_\beta & \tau_\beta^2 \end{pmatrix}$$

### Why Decompose?

1. **Interpretable priors.** We can put separate priors on the scales $\tau_\alpha, \tau_\beta$ (half-Cauchy or exponential) and on the correlation $\rho$ (LKJ prior), each with clear meaning.
2. **Prior independence.** We can express beliefs about "how much intercepts vary" independently from "how correlated intercepts and slopes are".
3. **Numerical stability.** The Cholesky decomposition (next section) works naturally with this factorisation.

---

## 4. The LKJ Prior on Correlation Matrices

We need a prior on $\boldsymbol{\Omega}$, a $K \times K$ correlation matrix (in our case $K = 2$).  The **LKJ distribution** (Lewandowski, Kurowicka, Joe, 2009) is the standard choice.

### Definition

$$\boldsymbol{\Omega} \sim \text{LKJCorr}(\eta)$$

The density is proportional to:

$$p(\boldsymbol{\Omega}) \propto |\boldsymbol{\Omega}|^{\eta - 1}$$

where $|\boldsymbol{\Omega}|$ is the determinant of the correlation matrix.

### What Does $\eta$ Control?

- $\eta = 1$: **Uniform** over all valid correlation matrices.  All correlations equally likely.
- $\eta > 1$: Concentrates mass toward the **identity matrix** ($\rho \to 0$).  Regularises toward independence.  The larger $\eta$, the stronger the pull toward zero correlation.
- $\eta < 1$: Concentrates mass toward **extreme correlations** ($|\rho| \to 1$).  Rarely used.

**Default recommendation:** $\eta = 2$.  This gently regularises toward independence without being too informative.

### For $K = 2$ (Our Case)

When there are only two correlated parameters (intercept and slope), $\boldsymbol{\Omega}$ is determined by a single correlation $\rho \in [-1, 1]$, and the LKJ prior simplifies to:

$$p(\rho) \propto (1 - \rho^2)^{\eta - 1}$$

For $\eta = 1$ this is uniform on $[-1, 1]$.  For $\eta = 2$ it is a symmetric Beta-like distribution peaked at $\rho = 0$.

The following plot shows the LKJ prior on $\rho$ for different values of $\eta$.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

rho_grid = np.linspace(-0.99, 0.99, 500)
for eta, col in [(1, "grey"), (2, "steelblue"), (4, "darkorange"), (8, "darkred")]:
    # LKJ for K=2: p(rho) ∝ (1 - rho^2)^(eta - 1)
    log_density = (eta - 1) * np.log(1 - rho_grid**2)
    density = np.exp(log_density - log_density.max())  # normalise for plotting
    density /= np.trapz(density, rho_grid)
    ax.plot(rho_grid, density, color=col, linewidth=2, label=f"$\\eta = {eta}$")

ax.set_xlabel(r"Correlation $\rho$")
ax.set_ylabel("Density")
ax.set_title("LKJ Prior on Correlation (K = 2)")
ax.legend(fontsize=11)
ax.set_xlim(-1, 1)
plt.tight_layout()
plt.show()

---

## 5. The Cholesky Decomposition

For MCMC efficiency, we do not sample $\boldsymbol{\Omega}$ directly.  Instead, we sample its **Cholesky factor** $\mathbf{L}$:

$$\boldsymbol{\Omega} = \mathbf{L} \mathbf{L}^\top$$

where $\mathbf{L}$ is a **lower-triangular matrix** with positive diagonal entries.

### Why Use the Cholesky Factor?

1. **Guaranteed positive-definiteness.** Any matrix of the form $\mathbf{L}\mathbf{L}^\top$ is automatically positive semi-definite.  The sampler can explore $\mathbf{L}$ freely without ever producing an invalid correlation matrix.
2. **Fewer constraints.** Sampling a symmetric PD matrix directly requires enforcing many constraints.  $\mathbf{L}$ has only $K(K+1)/2$ free parameters (the lower triangle).
3. **Efficient computation.** The multivariate normal $\boldsymbol{\alpha}_j \sim \text{MVN}(\boldsymbol{\gamma}, \boldsymbol{\Sigma})$ can be computed as $\boldsymbol{\alpha}_j = \boldsymbol{\gamma} + \text{diag}(\boldsymbol{\tau}) \cdot \mathbf{L} \cdot \mathbf{z}_j$ where $\mathbf{z}_j \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$.  This is the multivariate non-centered parameterisation.

### Non-Centered Parameterisation (Multivariate)

Combining the Cholesky factor with the non-centered trick:

$$\mathbf{z}_j \sim \mathcal{N}(\mathbf{0}, \mathbf{I}_K), \qquad \begin{pmatrix} \alpha_j \\ \beta_j \end{pmatrix} = \begin{pmatrix} \bar{\alpha} \\ \bar{\beta} \end{pmatrix} + \text{diag}(\boldsymbol{\tau}) \cdot \mathbf{L} \cdot \mathbf{z}_j$$

This is what we implement in PyMC below.

The following animation builds intuition for correlated varying slopes. Each line represents one group's regression. With only varying intercepts, the lines are parallel. When we add varying slopes with positive correlation, groups with high intercepts also tend to have steep slopes — the lines fan out.

In [ ]:
from manim import *

cfg.apply_manim_config()

In [ ]:
%%manim -qm -v WARNING VaryingSlopesAnimation

class VaryingSlopesAnimation(Scene):
    """Show varying intercepts (parallel) → varying slopes (fanning)."""

    def construct(self):
        # Axes
        axes = Axes(
            x_range=[-2, 2, 1], y_range=[-3, 5, 1],
            x_length=8, y_length=5,
            axis_config={"stroke_width": 1.5},
        ).shift(DOWN * 0.3)
        x_label = axes.get_x_axis_label("x", direction=RIGHT)
        y_label = axes.get_y_axis_label("y", direction=UP)
        self.play(Create(axes), Write(x_label), Write(y_label))

        # ── Phase 1: Varying intercepts only (parallel lines) ──
        title1 = Text("Varying Intercepts Only", font_size=24, color=BLUE).to_edge(UP, buff=0.3)
        self.play(Write(title1))

        J = 6
        np.random.seed(7)
        intercepts = np.linspace(-1.5, 2.5, J)
        shared_slope = 1.0
        colours = [RED, ORANGE, YELLOW, GREEN, TEAL, BLUE]

        lines_vi = []
        for j in range(J):
            line = axes.plot(
                lambda x, a=intercepts[j], b=shared_slope: a + b * x,
                x_range=[-1.8, 1.8], color=colours[j], stroke_width=2.5
            )
            lines_vi.append(line)

        self.play(*[Create(l) for l in lines_vi], run_time=1.5)

        note1 = Text("Same slope, different intercepts", font_size=16, color=GREY
        ).next_to(axes, DOWN, buff=0.3)
        self.play(Write(note1))
        self.wait(1.5)

        # ── Phase 2: Add varying slopes (positively correlated) ──
        title2 = Text("Varying Intercepts + Slopes", font_size=24, color=GREEN).to_edge(UP, buff=0.3)
        self.play(FadeOut(title1), FadeOut(note1), Write(title2))

        # Slopes correlated with intercepts (positive correlation)
        slopes = 0.3 + 0.4 * (intercepts - intercepts.mean()) / (intercepts.std() + 1e-8)
        slopes += np.random.normal(0, 0.15, J)

        lines_vs = []
        for j in range(J):
            line = axes.plot(
                lambda x, a=intercepts[j], b=slopes[j]: a + b * x,
                x_range=[-1.8, 1.8], color=colours[j], stroke_width=2.5
            )
            lines_vs.append(line)

        self.play(
            *[Transform(lines_vi[j], lines_vs[j]) for j in range(J)],
            run_time=2
        )

        note2 = Text("Lines fan out: slopes and intercepts are correlated",
                      font_size=16, color=GREY).next_to(axes, DOWN, buff=0.3)
        self.play(Write(note2))
        self.wait(1)

        # ── Phase 3: Show the correlation ellipse ──
        # Small inset showing (alpha_j, beta_j) pairs with correlation ellipse
        inset_ax = Axes(
            x_range=[-2.5, 3.5, 1], y_range=[-0.5, 1.5, 0.5],
            x_length=3, y_length=2,
            axis_config={"stroke_width": 1, "include_ticks": False},
        ).move_to(RIGHT * 4.5 + UP * 2.5)
        inset_label_x = cfg.math_text(r"\alpha_j", font_size=16).next_to(inset_ax.x_axis, DOWN, buff=0.05)
        inset_label_y = cfg.math_text(r"\beta_j", font_size=16).next_to(inset_ax.y_axis, LEFT, buff=0.05)

        self.play(Create(inset_ax), Write(inset_label_x), Write(inset_label_y))

        dots = VGroup(*[
            Dot(inset_ax.c2p(intercepts[j], slopes[j]), radius=0.06, color=colours[j])
            for j in range(J)
        ])
        self.play(*[GrowFromCenter(d) for d in dots])

        # Draw correlation ellipse
        from matplotlib.patches import Ellipse as MplEllipse
        cov = np.cov(intercepts, slopes)
        eigvals, eigvecs = np.linalg.eigh(cov)
        angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
        ellipse = Ellipse(width=2*np.sqrt(eigvals[1])*1.5, height=2*np.sqrt(eigvals[0])*1.5,
                          color=YELLOW, stroke_width=2, fill_opacity=0.1)
        ellipse.rotate(np.radians(angle))
        ellipse.move_to(inset_ax.c2p(intercepts.mean(), slopes.mean()))
        self.play(Create(ellipse))

        rho_val = np.corrcoef(intercepts, slopes)[0, 1]
        rho_text = cfg.math_text(f"\\rho \\approx {rho_val:.2f}", font_size=16
        ).next_to(inset_ax, DOWN, buff=0.1)
        self.play(Write(rho_text))
        self.wait(2)

In [ ]:
cfg.save_gifs(clean=True)

---

## 6. Worked Example: District-Level Contraception Use

We simulate data inspired by the Bangladesh Fertility Survey: women in $J$ districts decide whether to use contraception, and we model how the **urban/rural** effect on contraception varies by district.

The model:

$$\text{logit}(p_i) = \alpha_{j[i]} + \beta_{j[i]} \cdot \text{urban}_i$$

$$\begin{pmatrix} \alpha_j \\ \beta_j \end{pmatrix} \sim \text{MVNormal}\!\left(\begin{pmatrix} \bar{\alpha} \\ \bar{\beta} \end{pmatrix}, \boldsymbol{\Sigma}\right)$$

We use the decomposition $\boldsymbol{\Sigma} = \text{diag}(\boldsymbol{\tau}) \, \boldsymbol{\Omega} \, \text{diag}(\boldsymbol{\tau})$ with an LKJ prior on $\boldsymbol{\Omega}$.

In [ ]:
# ── Simulate Bangladesh-like data ──
J = 20          # districts
n_per_district = rng.integers(15, 80, J)  # unbalanced
N = n_per_district.sum()

# True hyperparameters
alpha_bar_true = -0.5   # baseline log-odds (about 38% contraception use)
beta_bar_true = 0.7     # average urban effect (positive: urban → more use)
tau_alpha_true = 0.6
tau_beta_true = 0.4
rho_true = -0.4         # negative correlation: high-baseline districts have weaker urban effect

# Covariance matrix
Sigma_true = np.array([
    [tau_alpha_true**2, rho_true * tau_alpha_true * tau_beta_true],
    [rho_true * tau_alpha_true * tau_beta_true, tau_beta_true**2]
])

# Sample group-level parameters
ab_true = rng.multivariate_normal(
    [alpha_bar_true, beta_bar_true], Sigma_true, size=J
)
alpha_true = ab_true[:, 0]
beta_true_j = ab_true[:, 1]

# Generate observations
district_idx = np.concatenate([np.full(n, j, dtype=int) for j, n in enumerate(n_per_district)])
urban = rng.binomial(1, 0.35, N).astype(float)  # ~35% urban

logit_p = alpha_true[district_idx] + beta_true_j[district_idx] * urban
p = 1 / (1 + np.exp(-logit_p))
y = rng.binomial(1, p)

print(f"Districts: {J}, Total observations: {N}")
print(f"Overall contraception use: {y.mean():.1%}")
print(f"True rho(alpha, beta) = {rho_true}")

In [ ]:
# ── Varying-slopes model in PyMC (non-centered + Cholesky) ──
if HAS_PYMC:
    with pm.Model() as varying_slopes_model:
        # Hyperpriors on population means
        gamma = pm.Normal("gamma", mu=0, sigma=1.5, shape=2)  # [alpha_bar, beta_bar]

        # Scales
        tau = pm.Exponential("tau", lam=1, shape=2)  # [tau_alpha, tau_beta]

        # Correlation matrix (Cholesky factor)
        chol, corr, stds = pm.LKJCholeskyCov(
            "chol_cov",
            n=2,
            eta=2.0,
            sd_dist=pm.Exponential.dist(lam=1),
            compute_corr=True,
        )

        # Non-centered group effects
        z = pm.Normal("z", mu=0, sigma=1, shape=(J, 2))
        ab = pm.Deterministic("ab", gamma + pm.math.dot(z, chol.T))

        alpha = ab[:, 0]
        beta_j = ab[:, 1]

        # Likelihood
        logit_p_obs = alpha[district_idx] + beta_j[district_idx] * urban
        obs = pm.Bernoulli("obs", logit_p=logit_p_obs, observed=y)

        # Sample
        trace_vs = pm.sample(2000, tune=1500, chains=4, random_seed=42,
                             target_accept=0.95, return_inferencedata=True)

    print(az.summary(trace_vs, var_names=["gamma", "chol_cov_corr"]))
else:
    print("PyMC not available.")

### 6.1 Interpreting the Correlation

In [ ]:
if HAS_PYMC:
    # Extract posterior correlation
    corr_samples = trace_vs.posterior["chol_cov_corr"].values[:, :, 0, 1].flatten()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: posterior of rho
    ax = axes[0]
    ax.hist(corr_samples, bins=50, density=True, color="steelblue", alpha=0.7)
    ax.axvline(rho_true, color="red", linestyle="--", linewidth=2,
               label=f"True $\\rho$ = {rho_true}")
    ax.axvline(corr_samples.mean(), color="gold", linewidth=2,
               label=f"Posterior mean = {corr_samples.mean():.2f}")
    ax.set_xlabel(r"$\rho(\alpha, \beta)$")
    ax.set_ylabel("Density")
    ax.set_title("Posterior of Intercept–Slope Correlation")
    ax.legend(fontsize=10)

    # Right: group-level (alpha_j, beta_j) scatter with posterior ellipse
    ax = axes[1]
    ab_post = trace_vs.posterior["ab"].values.reshape(-1, J, 2)
    ab_post_mean = ab_post.mean(axis=0)

    ax.scatter(alpha_true, beta_true_j, color="red", s=60, zorder=5,
               label="True $(\\alpha_j, \\beta_j)$", marker="x")
    ax.scatter(ab_post_mean[:, 0], ab_post_mean[:, 1], color="green", s=60,
               zorder=5, label="Posterior mean")

    # Arrows from true to estimated
    for j in range(J):
        ax.annotate(
            "", xy=(ab_post_mean[j, 0], ab_post_mean[j, 1]),
            xytext=(alpha_true[j], beta_true_j[j]),
            arrowprops=dict(arrowstyle="->", color="grey", lw=0.8, alpha=0.5)
        )

    ax.set_xlabel(r"$\alpha_j$ (intercept)")
    ax.set_ylabel(r"$\beta_j$ (urban effect)")
    ax.set_title("Group-Level Intercepts vs. Slopes")
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

The left panel shows the posterior of $\rho$, which should concentrate around the true value of $-0.4$. The negative correlation means that districts with high baseline contraception use ($\alpha_j$ large) tend to have a *weaker* urban effect ($\beta_j$ small) — the urban/rural gap is smaller where overall use is already high.

The right panel shows each district's intercept–slope pair. The posterior means (green) are shrunk toward the population centre relative to the true values (red crosses), with small districts shrinking more.

### 6.2 Visualising Group-Level Regression Lines

In [ ]:
if HAS_PYMC:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    x_plot = np.array([0, 1])  # rural, urban

    # Left: No-pooling (raw) regression lines
    ax = axes[0]
    ax.set_title("No Pooling (Raw Estimates)")
    for j in range(J):
        mask = district_idx == j
        p_rural = y[mask & (urban == 0)].mean() if (mask & (urban == 0)).sum() > 0 else 0.5
        p_urban = y[mask & (urban == 1)].mean() if (mask & (urban == 1)).sum() > 0 else 0.5
        ax.plot(x_plot, [p_rural, p_urban], color="red", alpha=0.4, linewidth=1)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Rural", "Urban"])
    ax.set_ylabel("P(contraception)")
    ax.set_ylim(-0.05, 1.05)

    # Right: Partial-pooling regression lines
    ax = axes[1]
    ax.set_title("Partial Pooling (Hierarchical Model)")
    for j in range(J):
        a, b = ab_post_mean[j]
        p_vals = 1 / (1 + np.exp(-(a + b * x_plot)))
        ax.plot(x_plot, p_vals, color="green", alpha=0.5, linewidth=1.5)

    # Population mean
    gamma_post_mean = trace_vs.posterior["gamma"].values.reshape(-1, 2).mean(axis=0)
    p_pop = 1 / (1 + np.exp(-(gamma_post_mean[0] + gamma_post_mean[1] * x_plot)))
    ax.plot(x_plot, p_pop, color="gold", linewidth=3, label="Population mean")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Rural", "Urban"])
    ax.set_ylabel("P(contraception)")
    ax.set_ylim(-0.05, 1.05)
    ax.legend()

    plt.suptitle("District-Level Regression Lines: No Pooling vs. Partial Pooling", fontsize=13)
    plt.tight_layout()
    plt.show()

The no-pooling lines (left) are noisy — some show extreme or even reversed effects due to small samples.  The partial-pooling lines (right) are regularised: they cluster around the population mean (gold), with districts that have more data deviating further.  This is the multivariate generalisation of shrinkage.

---

## 7. Generalising to $K$ Varying Coefficients

Everything above extends naturally from 2 to $K$ varying coefficients:

$$\boldsymbol{\theta}_j = (\alpha_j, \beta_{1j}, \ldots, \beta_{(K-1)j})^\top \sim \text{MVNormal}(\boldsymbol{\gamma}, \boldsymbol{\Sigma})$$

with $\boldsymbol{\Sigma} = \text{diag}(\boldsymbol{\tau}) \, \boldsymbol{\Omega} \, \text{diag}(\boldsymbol{\tau})$ and $\boldsymbol{\Omega} \sim \text{LKJ}(\eta)$.

The Cholesky factor $\mathbf{L}$ is now $K \times K$ lower-triangular with $K(K+1)/2$ free parameters.

**Practical considerations for larger $K$:**

| $K$ | Cholesky parameters | Notes |
|-----|---------------------|-------|
| 2   | 3                   | Single correlation — easy |
| 3   | 6                   | Still manageable |
| 5   | 15                  | Getting complex |
| 10  | 55                  | Consider regularisation ($\eta \geq 2$) |

For large $K$, the number of correlation parameters grows quadratically.  Strong regularisation via the LKJ prior ($\eta = 4$ or higher) helps prevent overfitting.  In extreme cases, a diagonal approximation ($\rho = 0$) may be needed, sacrificing the ability to learn correlations.

---

## 8. Key Takeaways

1. **Varying slopes** extend the varying-intercept model by allowing the *effect* of predictors to differ across groups, not just the baseline.

2. The group-level intercepts and slopes follow a **multivariate normal** distribution.  The covariance matrix captures both the spread and the **correlation** between group-level parameters.

3. We decompose $\boldsymbol{\Sigma} = \text{diag}(\boldsymbol{\tau}) \, \boldsymbol{\Omega} \, \text{diag}(\boldsymbol{\tau})$ to separate scales from correlations, enabling independent priors on each.

4. The **LKJ prior** on $\boldsymbol{\Omega}$ controls regularisation toward independence.  $\eta = 2$ is a sensible default.

5. The **Cholesky decomposition** guarantees positive-definiteness and enables the non-centered parameterisation in the multivariate case.

6. The learned correlations carry substantive meaning — e.g., whether high-baseline groups respond differently to treatments.

## Further Reading

- Lewandowski, D., Kurowicka, D. & Joe, H. (2009). *Generating random correlation matrices based on vines and extended onion method*. Journal of Multivariate Analysis.
- McElreath, R. (2020). *Statistical Rethinking*, Ch. 14 (Adventures in covariance).
- Gelman & Hill (2007). *Data Analysis Using Regression and Multilevel/Hierarchical Models*, Chs. 12–13.

**Next notebook:** We put it all together in an end-to-end analysis, including model comparison, the Mundlak device for group-level confounding, and practical workflow advice.